# Inspect environmental evidence and actions

**Question:** Which environmental values have usable semantics and coverage, and which decisions remain blocked?

Read-only inspection of native lunar crops, the unit sidecar, climate summaries and Diviner region/bin summaries. Saved run outputs are hash-checked before use. This notebook does not re-read all climate or Diviner source rows; notebook 07 performed that audit. Summary-level availability cannot establish per-patch thermal coverage. No cross-body score or arbitrary temperature cutoff is introduced.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
PREFIX = 'environment-inspection-'
TITLE = 'Environmental inspection: conclusions and actions'
NOTEBOOK = 'preprocessing/11_inspect_environment_and_actions.ipynb'
import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from IPython.display import HTML, SVG, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import bar_svg, check_input, file_hash, verify_run

OUT = ROOT / 'notebooks/diagnostics' / (PREFIX + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
evidence = {}
expected = {}
actions = []

def register(stage, run_id):
    run = ROOT / 'data/processed' / stage / run_id
    expected.update(verify_run(run))
    evidence[str(run / 'provenance.json')] = file_hash(run / 'provenance.json')
    return run

def checked(path):
    return check_input(path, expected, evidence)

def read_layer(path):
    with rasterio.open(checked(path)) as src:
        assert src.count == 1 and src.scales == (1.0,) and src.offsets == (0.0,), path
        return src.read(1, masked=True).astype('float32').filled(np.nan), (src.shape, src.transform, src.crs)

def chart(labels, values, title, unit, filename):
    svg = bar_svg(list(labels), [float(v) for v in values], title, unit)
    (OUT / filename).write_text(svg, encoding='utf-8')
    display(SVG(svg))


In [ ]:
CONFIG = {'native_units_policy': 'documented only', 'illumination_range_tolerance': 1e-6,
          'diviner_coverage': 'region-bin only; not per-cell temporal completeness'}
tolerance = CONFIG['illumination_range_tolerance']
assert np.isfinite(tolerance) and tolerance >= 0
moon = register('lunar_terrain_environment', '20261028T022855014566Z_cffea7')
climate_run = register('earth_climate', '20261028T014917410405Z_d7c8d5')
diviner_run = register('diviner_temperature', '20261028T022942126792Z_d44299')
unit_path = ROOT / 'notebooks/diagnostics/environment-readiness-20261028/native_layer_units.json'
units = pd.read_json(unit_path, encoding='utf-8-sig')
evidence[str(unit_path)] = file_hash(unit_path)
assert len(units) == 273 and not units.duplicated(['region', 'file']).any()
print('Unit sidecar is an interpretation; source TIFF tags remain unchanged.')


## Native lunar layers: unit gate and measured ranges

Every native crop is read. A documented unit permits interpretation, not automatic inclusion in a model. Range checks are applied only to families with explicit semantics. Other products retain a “not assessed” range status; plausible values do not resolve missing documentation. Partial valid coverage must be intersected with the intended analysis footprint later.

In [ ]:
native_rows = []
for item in units.to_dict('records'):
    paths = list((moon / 'moon' / item['region'] / 'native').glob('*/' + item['file']))
    assert len(paths) == 1, item
    values, _ = read_layer(paths[0])
    valid = values[np.isfinite(values)]
    low, high = (float(valid.min()), float(valid.max())) if valid.size else (None, None)
    range_status = 'not_assessed'
    if valid.size and item['file'].startswith(('AVGVISIB', 'LPSR', 'SKYV')):
        if item['file'].startswith('LPSR'):
            okay = bool(np.isin(valid, [0, 1]).all())
        else:
            ceiling = 4 * np.pi if item['file'].startswith('SKYV') else 1
            okay = bool(low >= -tolerance and high <= ceiling + tolerance)
        range_status = 'pass' if okay else 'fail'
    native_rows.append({'region': item['region'], 'file': item['file'], 'unit': item['unit'],
                            'unit_status': item['status'], 'valid_fraction': valid.size / values.size,
                            'minimum': low, 'maximum': high, 'range_check': range_status})
native = pd.DataFrame(native_rows)
native.to_json(OUT / 'native_checks.json', orient='records', indent=2)
display(native.groupby(['unit_status', 'range_check']).size().rename('crops').to_frame())
display(native.sort_values('valid_fraction').head(20))
counts = native.groupby('unit_status').size()
chart(counts.index, counts.values, 'Native crop unit evidence', 'crop count', 'unit_evidence.svg')
blocked = native.loc[(native.unit_status != 'documented') | (native.range_check == 'fail') | (native.valid_fraction == 0)]
blocked.to_json(OUT / 'held_native_layers.json', orient='records', indent=2)
actions.append(f'{len(blocked)} native crops held from quantitative combinations: resolve their unit/variant evidence or range/empty failures; see held_native_layers.json.')
actions.append(f'{int((native.range_check == "fail").sum())} explicit illumination range failures. Passing values still require spatial registration checks before terrain joins.')


## Diviner: coverage across seasonal/subsolar bins

These bins are not monthly dates. A bin is “available” here if it contains at least one valid temperature cell somewhere in the region. Even a common bin across all regions need not cover the same locations. Spatial extrema of binned averages are not mission temperature limits.

In [ ]:
thermal = pd.read_csv(checked(diviner_run / 'regional_bin_summary.csv'))
keys = ['region', 'seasonal_ls_bin', 'subsolar_longitude_bin']
assert len(thermal) == 1008 and not thermal.duplicated(keys).any()
assert thermal.groupby('region').size().eq(144).all()
assert (thermal.valid_temperature_rows >= 0).all()
assert (thermal.valid_temperature_rows <= thermal.selected_rows).all()
matrix = thermal.pivot(index='region', columns='source_product', values='valid_temperature_rows')
assert matrix.shape == (7, 144) and matrix.notna().all().all()
available = matrix > 0
common = available.columns[available.all(axis=0)].tolist()
summary = pd.DataFrame({'available_bins': available.sum(axis=1), 'empty_bins': (~available).sum(axis=1)})
display(summary)
display(thermal.nsmallest(10, 'spatial_min_of_bin_average_K')[keys + ['spatial_min_of_bin_average_K']])
chart(summary.index, summary.empty_bins, 'Diviner empty region bins', 'empty bins out of 144', 'diviner_empty_bins.svg')
# Plain pandas HTML needs no optional styling dependency.
matrix_html = available.astype(int).to_html().replace('<td>1</td>', '<td style="background:#bde3d1">1</td>')
matrix_html = matrix_html.replace('<td>0</td>', '<td style="background:#f3bec7">0</td>')
matrix_html = '<p>1 / green = at least one valid cell; 0 / pink = empty. Columns are source-product IDs.</p><div style="overflow:auto;max-height:350px">' + matrix_html + '</div>'
(OUT / 'diviner_matrix.html').write_text(matrix_html, encoding='utf-8')
display(HTML(matrix_html))
(OUT / 'common_region_bins.json').write_text(json.dumps(common, indent=2), encoding='utf-8')
summary.to_json(OUT / 'diviner_coverage.json', orient='index', indent=2)
actions.append(f'{int((~available).to_numpy().sum())} empty region/bin pairs remain missing. '
               f'{len(common)} bins have at least one valid cell in every region; use common_region_bins.json only as a starting set, then check cell-level coverage.')
actions.append('Hold thermal hazard limits: inspect low-end retrieval quality against source flags; do not convert missing bins to zero or impose an unsupported temperature cutoff.')


## OSM support: export completeness limits

Inspect the saved quality summaries, including incomplete geometry counts. These are export diagnostics, not a fresh geometry audit or a measure of real-world mapping completeness. No site access or proximity claim follows from feature counts. Attribution: © OpenStreetMap contributors; Geofabrik extracts, ODbL 1.0.

In [ ]:
osm_run = register('osm_support', '20261028T015656094003Z_ec31c5')
osm_rows = []
for path in sorted((osm_run / 'earth').glob('*/quality.json')):
    quality = json.loads(checked(path).read_text())
    assert quality['feature_count'] == sum(quality['categories'].values())
    osm_rows.append({'region': path.parent.name, 'exported_features': quality['feature_count'],
                     'incomplete_or_invalid_areas': quality['diagnostics'].get('incomplete_or_invalid_area', 0),
                     'limitations': quality['limitations']})
assert len(osm_rows) == 3
display(pd.DataFrame(osm_rows))
(OUT / 'osm_quality_checks.json').write_text(json.dumps(osm_rows, indent=2), encoding='utf-8')
actions.append(f"OSM quality summaries report {sum(r['exported_features'] for r in osm_rows)} exported features and "
               f"{sum(r['incomplete_or_invalid_areas'] for r in osm_rows)} incomplete/invalid areas. "
               'Retain source limitations and verify access independently; no infrastructure completeness claim.')


## Earth climate: temporal completeness and spatial support

Checks all expected region/month/variable summary keys, units, and coverage. Temporal completeness and geographic coverage are separate. Lanzarote's bounding box includes ocean; the earlier land-aware audit is needed to interpret low box coverage. No monthly value is interpreted as a daily extreme.

In [ ]:
climate = pd.read_csv(checked(climate_run / 'regional_monthly_summary.csv'), parse_dates=['date'])
expected_index = pd.MultiIndex.from_product([
    ['northern_arizona', 'lanzarote', 'atacama_san_pedro'],
    pd.date_range('2015-01-01', '2024-12-01', freq='MS'), ['tmin', 'tmax', 'ppt', 'ws', 'srad']],
    names=['area', 'date', 'variable'])
actual_index = pd.MultiIndex.from_frame(climate[['area', 'date', 'variable']])
assert actual_index.is_unique and len(actual_index) == len(expected_index)
assert expected_index.difference(actual_index).empty
assert climate.weighted_coverage.between(0, 1).all()
assert climate.groupby('variable').units.nunique().eq(1).all()
assert np.isfinite(climate.spatial_weighted_mean).all()
climate_summary = climate.groupby(['area', 'variable', 'units']).agg(
    months=('date', 'size'), minimum_box_coverage=('weighted_coverage', 'min'),
    minimum_monthly_regional_mean=('spatial_weighted_mean', 'min'),
    maximum_monthly_regional_mean=('spatial_weighted_mean', 'max')).reset_index()
display(climate_summary)
climate_summary.to_json(OUT / 'climate_summary_checks.json', orient='records', indent=2)
cover = climate.groupby('area').weighted_coverage.min() * 100
chart(cover.index, cover.values, 'Minimum native climate box coverage', 'percent of weighted selected grid', 'climate_coverage.svg')
actions.append(f'All {len(climate)} climate summary keys are present with finite regional means. Retain native units and monthly aggregation; do not upscale these into independent 30 m observations.')
actions.append('For Lanzarote, retain coastal/ocean missingness. Prior land-support analysis found about 99.954% of mapped land represented across selected cell footprints; this is not exact-ROI completeness.')
actions.append('Proceed with terrain-only inspection using notebook 10. Use documented environmental layers as separate context; defer combined suitability scoring and validate registration before spatial joins.')
# Commit a completion marker only after all checks and exports succeed.
report = '# ' + TITLE + '\n\n' + '\n'.join('- ' + item for item in actions)
report += '\n\nThese are data-readiness conclusions, not scientific accuracy or site-safety certification. '
report += 'Inputs were read-only. Consult the tables and figures in this folder for measurements.\n'
(OUT / 'REPORT.md').write_text(report, encoding='utf-8')
evidence[str(ROOT / 'notebooks' / NOTEBOOK)] = file_hash(ROOT / 'notebooks' / NOTEBOOK)
evidence[str(ROOT / 'notebooks/inspection.py')] = file_hash(ROOT / 'notebooks/inspection.py')
(OUT / 'provenance.json').write_text(json.dumps({
    'status': 'complete', 'inputs_sha256': evidence, 'configuration': CONFIG,
    'versions': {'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__, 'rasterio': rasterio.__version__},
    'outputs_sha256': {p.name: file_hash(p) for p in OUT.iterdir() if p.is_file()},
}, indent=2), encoding='utf-8')
display(Markdown(report))
print('Saved:', OUT)
